# Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Decision window:** March 2026  
**Outcome window:** April 2026

This notebook builds a transparent, hand-written baseline for prioritizing content for review. The score uses March-only information. April is used only after scoring, for outcome evaluation.

The assignment requires two signal checks, at least one linked to a real FlyRank flag, one transparent score with reason codes and an action label, a ranked queue, a top-10 review, a weak-pick review, and a leakage check.


## 1. Rule, signal tests, and reason codes

### Rule in plain English

1. **Volume / quick-win signal:** prioritize content with at least **500 March GSC impressions** so the review queue focuses on content with enough observed search visibility to matter.
2. **CTR-fix signal:** among visible pages with positions **4–20**, flag pages whose March CTR is below the **March-only median CTR** for that same visible position band. This is the flag-linked signal: it tests the intuition behind FlyRank's **CTR-vs-position / CTR-fix** logic.
3. **Score:** 5 points when both signals fire, 3 points for high volume only, and 0 otherwise.

The score is deliberately not fitted to April outcomes. The only values used to construct the score are March observations and a March-only CTR cutoff.

### Reason codes

- `high_volume_ctr_fix` — enough March visibility and a CTR-fix pattern.
- `high_volume_only` — enough March visibility, but no CTR-fix pattern.
- `monitor` — neither rule condition fired.

### Actions

- `CTR-fix review` — inspect title/snippet/intent alignment and consider a CTR improvement test.
- `Quick-win review` — inspect the page because it has meaningful search visibility.
- `Monitor` — no baseline action priority.


In [1]:
# Setup: connect to the FlyRank warehouse

%pip -q install duckdb pandas numpy

import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add your Read token to Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(
    "CREATE OR REPLACE SECRET hf_token "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN],
)

MARCH_REL = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

APRIL_REL = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet"
)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Warehouse connection configured.")
print("Decision window: March 2026")
print("Outcome window: April 2026")


Warehouse connection configured.
Decision window: March 2026
Outcome window: April 2026


In [2]:
# Verify the real March schema before querying features.

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{MARCH_REL}')"
).df()

display(schema[["column_name", "column_type"]])

required_columns = {
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "month",
}

missing = sorted(required_columns - set(schema["column_name"]))
assert not missing, f"Required warehouse columns are missing: {missing}"

print("PASS: required March warehouse fields are present.")


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


PASS: required March warehouse fields are present.


In [3]:
# Build the decision/evaluation frame in one remote query.
#
# Important:
# - March fields are the only inputs available to the baseline score.
# - April is joined only to create the future evaluation label.
# - GSC availability is filtered with IS TRUE, matching the data contract.
# - Average position is impression-weighted, matching the established data contract.
# - The future label is the established proxy: April impressions >20% below March,
#   with positive March impressions.

evaluation_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS march_ctr_pct,
        SUM(
            CASE
                WHEN gsc_impressions > 0
                     AND gsc_avg_position > 0
                THEN gsc_impressions * gsc_avg_position
                ELSE 0
            END
        ) / NULLIF(
            SUM(
                CASE
                    WHEN gsc_impressions > 0
                         AND gsc_avg_position > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS march_avg_position,
        COUNT(
            DISTINCT CASE
                WHEN gsc_impressions > 0 THEN report_date
            END
        ) AS march_impression_days
    FROM read_parquet('{MARCH_REL}')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet('{APRIL_REL}')
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_ctr_pct,
    m.march_avg_position,
    m.march_impression_days,
    a.april_impressions,
    CASE
        WHEN m.march_impressions > 0
             AND a.april_impressions < 0.80 * m.march_impressions
        THEN 1
        ELSE 0
    END AS future_decline_label
FROM march AS m
INNER JOIN april AS a
    USING (client_hash_id, content_hash_id)
"""

evaluation_frame = con.execute(evaluation_sql).df()

assert evaluation_frame[["client_hash_id", "content_hash_id"]].duplicated().sum() == 0

print(f"Decision/evaluation rows: {len(evaluation_frame):,}")
print(f"Future decline rate: {evaluation_frame['future_decline_label'].mean():.4f}")
display(evaluation_frame.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision/evaluation rows: 158,549
Future decline rate: 0.4782


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,april_impressions,future_decline_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.175439,4.450877,31,1151.0,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,6.894737,26,73.0,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,6.363636,30,98.0,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.422238,6.906404,31,2275.0,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.577617,3.950542,31,6266.0,0
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,0.0,0.000000,18.934783,21,98.0,0
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,0.666667,7.929577,30,100.0,1
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.0,0.000000,17.866667,26,37.0,1
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,0.380291,4.931878,31,8511.0,0
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,0.000000,61.815315,30,117.0,1


### Data-contract check

The future label above is **evaluation-only**. It is not an input to the baseline score.

`future_decline_label = 1` means April GSC impressions were more than 20% below March GSC impressions, with a positive March baseline. This is the established future-outcome proxy from the preceding data-contract work.

The notebook therefore keeps the decision boundary clean:

**March observations → score/rank → then April outcome evaluation.**


In [4]:
# Signal test 1 — Volume / quick-win signal
#
# Visible bucket table: five March-impression buckets with n and future decline rate.
# The baseline signal itself is the fixed, decision-time threshold of 500 impressions.

signal1 = evaluation_frame.copy()

signal1["volume_bucket"] = pd.qcut(
    signal1["march_impressions"].rank(method="first"),
    q=5,
    labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"],
)

volume_buckets = (
    signal1.groupby("volume_bucket", observed=False)
    .agg(
        n=("future_decline_label", "size"),
        decline_rate=("future_decline_label", "mean"),
        median_march_impressions=("march_impressions", "median"),
    )
    .reset_index()
)

volume_buckets["decline_rate_pct"] = (100 * volume_buckets["decline_rate"]).round(2)

volume_threshold = 500
high_volume = signal1["march_impressions"] >= volume_threshold
volume_n = int(high_volume.sum())
volume_rate = signal1.loc[high_volume, "future_decline_label"].mean()
base_rate = signal1["future_decline_label"].mean()

if volume_n == 0:
    volume_verdict = "FALSE"
elif volume_rate >= base_rate + 0.05:
    volume_verdict = "CONFIRMED"
elif volume_rate <= base_rate - 0.05:
    volume_verdict = "OPPOSITE"
else:
    volume_verdict = "MIXED"

print("Signal 1: Volume behind the quick-win idea")
display(
    volume_buckets[
        ["volume_bucket", "n", "decline_rate_pct", "median_march_impressions"]
    ]
)
print(f"Fixed threshold: March impressions >= {volume_threshold:,}")
print(f"n above threshold: {volume_n:,}")
print(f"Decline rate above threshold: {100 * volume_rate:.2f}%")
print(f"Overall base rate: {100 * base_rate:.2f}%")
print(f"VERDICT: {volume_verdict}")


Signal 1: Volume behind the quick-win idea


,volume_bucket,n,decline_rate_pct,median_march_impressions
0,Q1 lowest,31710,35.11,7.0
1,Q2,31710,49.94,61.0
2,Q3,31709,53.62,246.0
3,Q4,31710,53.13,895.0
4,Q5 highest,31710,47.28,4427.5


Fixed threshold: March impressions >= 500
n above threshold: 61,846
Decline rate above threshold: 49.99%
Overall base rate: 47.82%
VERDICT: MIXED


In [5]:
# Signal test 2 — CTR-vs-position / CTR-fix signal
#
# This is the required flag-linked check.
# We first restrict to visible pages in positions 4–20, then define the CTR cutoff
# using only March data: the median CTR in that March-only visible band.

ctr_test = evaluation_frame.copy()

visible_position_band = (
    ctr_test["march_impressions"].ge(500)
    & ctr_test["march_avg_position"].between(4, 20, inclusive="both")
    & ctr_test["march_ctr_pct"].notna()
)

ctr_cutoff = ctr_test.loc[visible_position_band, "march_ctr_pct"].median()
assert pd.notna(ctr_cutoff), "CTR cutoff could not be computed from March data."

band = ctr_test.loc[visible_position_band].copy()
band["ctr_position_bucket"] = pd.qcut(
    band["march_ctr_pct"].rank(method="first"),
    q=5,
    labels=["Q1 lowest CTR", "Q2", "Q3", "Q4", "Q5 highest CTR"],
)

ctr_buckets = (
    band.groupby("ctr_position_bucket", observed=False)
    .agg(
        n=("future_decline_label", "size"),
        decline_rate=("future_decline_label", "mean"),
        median_ctr_pct=("march_ctr_pct", "median"),
        median_position=("march_avg_position", "median"),
    )
    .reset_index()
)

ctr_buckets["decline_rate_pct"] = (100 * ctr_buckets["decline_rate"]).round(2)

ctr_fix_candidate = (
    visible_position_band
    & ctr_test["march_ctr_pct"].lt(ctr_cutoff)
)

ctr_fix_n = int(ctr_fix_candidate.sum())
ctr_fix_rate = ctr_test.loc[ctr_fix_candidate, "future_decline_label"].mean()

if ctr_fix_n == 0:
    ctr_verdict = "FALSE"
elif ctr_fix_rate >= base_rate + 0.05:
    ctr_verdict = "CONFIRMED"
elif ctr_fix_rate <= base_rate - 0.05:
    ctr_verdict = "OPPOSITE"
else:
    ctr_verdict = "MIXED"

print("Signal 2: CTR-vs-position behind CTR-fix logic")
display(
    ctr_buckets[
        [
            "ctr_position_bucket",
            "n",
            "decline_rate_pct",
            "median_ctr_pct",
            "median_position",
        ]
    ]
)
print("Visible position band: positions 4–20 with >=500 March impressions")
print(f"March-only CTR cutoff (median): {ctr_cutoff:.3f}%")
print(f"n below CTR cutoff: {ctr_fix_n:,}")
print(f"Decline rate below cutoff: {100 * ctr_fix_rate:.2f}%")
print(f"Overall base rate: {100 * base_rate:.2f}%")
print(f"VERDICT: {ctr_verdict}")


Signal 2: CTR-vs-position behind CTR-fix logic


,ctr_position_bucket,n,decline_rate_pct,median_ctr_pct,median_position
0,Q1 lowest CTR,7252,63.97,0.000000,8.313263
1,Q2,7251,57.95,0.100220,7.216532
2,Q3,7251,51.54,0.190840,6.990895
3,Q4,7251,38.97,0.348584,6.490494
4,Q5 highest CTR,7251,32.27,0.676745,6.970395


Visible position band: positions 4–20 with >=500 March impressions
March-only CTR cutoff (median): 0.191%
n below CTR cutoff: 18,120
Decline rate below cutoff: 59.54%
Overall base rate: 47.82%
VERDICT: CONFIRMED


### Signal verdicts

The two verdicts above are deliberately evidence-led:

- **Volume / quick-win:** the fixed 500-impression threshold is compared with the overall future-decline base rate.
- **CTR-vs-position / CTR-fix:** the March-only CTR cutoff is tested against the same future outcome.

A `CONFIRMED` result means the tested group is at least 5 percentage points above the base rate in this development window. `OPPOSITE` means it is at least 5 points below. `MIXED` means the observed difference is smaller. `FALSE` means there were no usable rows.

A negative verdict is acceptable: it is evidence that a rule assumption should not be trusted blindly.


## 2. Build the ranked queue

The score is now **frozen**. No April value and no future label is used to construct it.

### Frozen baseline

- `high_volume = march_impressions >= 500`
- `ctr_fix = high_volume AND position in [4, 20] AND march_ctr_pct < March-only median CTR for that visible band`
- `score = 5` when both fire
- `score = 3` when only high volume fires
- `score = 0` otherwise

This is intentionally simple enough for a non-engineer to audit.


In [6]:
# Freeze the transparent March-only rule and create reason codes.

scored = evaluation_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "march_ctr_pct",
        "march_avg_position",
        "march_impression_days",
    ]
].copy()

scored["high_volume"] = scored["march_impressions"] >= volume_threshold

scored["ctr_fix"] = (
    scored["high_volume"]
    & scored["march_avg_position"].between(4, 20, inclusive="both")
    & scored["march_ctr_pct"].notna()
    & scored["march_ctr_pct"].lt(ctr_cutoff)
)

scored["score"] = np.select(
    [
        scored["high_volume"] & scored["ctr_fix"],
        scored["high_volume"],
    ],
    [5, 3],
    default=0,
).astype(int)

scored["reason_code"] = np.select(
    [
        scored["high_volume"] & scored["ctr_fix"],
        scored["high_volume"],
    ],
    ["high_volume_ctr_fix", "high_volume_only"],
    default="monitor",
)

scored["action"] = np.select(
    [
        scored["reason_code"].eq("high_volume_ctr_fix"),
        scored["reason_code"].eq("high_volume_only"),
    ],
    ["CTR-fix review", "Quick-win review"],
    default="Monitor",
)

ranked_queue = scored.sort_values(
    ["score", "march_impressions", "march_clicks"],
    ascending=[False, False, False],
    kind="mergesort",
).reset_index(drop=True)

ranked_queue.insert(0, "rank", np.arange(1, len(ranked_queue) + 1))

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_impression_days",
]

ranked_queue = ranked_queue[queue_columns].copy()

csv_path = OUTPUT_DIR / "baseline_action_score.csv"
ranked_queue.to_csv(csv_path, index=False)

print(f"Ranked queue rows: {len(ranked_queue):,}")
print(f"CSV written to: {csv_path}")
display(ranked_queue.head(10))


Ranked queue rows: 158,549
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,reason_code,action,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days
0,1,client_62f4a7e64f5e0096,content_b99ea6861864dea5,5,high_volume_ctr_fix,CTR-fix review,194337.0,361.0,0.185760,4.551516,31
1,2,client_73cda7b4e4f265ea,content_f43118e089ecc69a,5,high_volume_ctr_fix,CTR-fix review,139417.0,191.0,0.136999,5.342218,31
2,3,client_62f4a7e64f5e0096,content_7c6373141eae744a,5,high_volume_ctr_fix,CTR-fix review,132593.0,83.0,0.062598,5.948459,31
3,4,client_73cda7b4e4f265ea,content_95ff62babbfac9c7,5,high_volume_ctr_fix,CTR-fix review,122857.0,234.0,0.190465,4.623098,31
4,5,client_23a62021009f63c4,content_5e1c049f62e33b11,5,high_volume_ctr_fix,CTR-fix review,120175.0,168.0,0.139796,17.773364,31
5,6,client_73cda7b4e4f265ea,content_e578ac84778da489,5,high_volume_ctr_fix,CTR-fix review,117764.0,163.0,0.138412,4.203950,31
6,7,client_62f4a7e64f5e0096,content_f6116743b00afc2d,5,high_volume_ctr_fix,CTR-fix review,107584.0,15.0,0.013943,9.735658,31
7,8,client_e547b89c05043229,content_21309e9a83c83653,5,high_volume_ctr_fix,CTR-fix review,103187.0,192.0,0.186070,5.036739,29
8,9,client_73cda7b4e4f265ea,content_cf651123f1085418,5,high_volume_ctr_fix,CTR-fix review,101363.0,175.0,0.172647,6.286308,31
9,10,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,5,high_volume_ctr_fix,CTR-fix review,89332.0,4.0,0.004478,7.831807,31


### Baseline evaluation

Precision@K answers the actual ranking question:

> Of the first K items the rule prioritizes, what fraction later matched the future decline outcome?

The base rate is printed beside it so the result has context.


In [7]:
# Evaluate the frozen queue against the April-only outcome.
#
# The outcome is merged only after ranking, so it cannot influence score construction.

evaluation_lookup = evaluation_frame[
    ["client_hash_id", "content_hash_id", "future_decline_label"]
].copy()

ranked_eval = ranked_queue.merge(
    evaluation_lookup,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one",
)

def precision_at_k(df, k):
    top = df.head(k)
    if len(top) == 0:
        return np.nan
    return float(top["future_decline_label"].mean())

ks = [10, 50, 100, 500]
precision_rows = []

for k in ks:
    precision_rows.append(
        {
            "k": k,
            "n_in_top_k": min(k, len(ranked_eval)),
            "precision_at_k": precision_at_k(ranked_eval, k),
        }
    )

precision_table = pd.DataFrame(precision_rows)
precision_table["precision_pct"] = (100 * precision_table["precision_at_k"]).round(2)

print(f"Overall future-decline base rate: {100 * base_rate:.2f}%")
display(
    precision_table[
        ["k", "n_in_top_k", "precision_at_k", "precision_pct"]
    ]
)

metrics = {
    "decision_rows": int(len(evaluation_frame)),
    "future_decline_rate": float(base_rate),
    "volume_threshold": int(volume_threshold),
    "volume_signal_verdict": volume_verdict,
    "ctr_position_band": [4, 20],
    "ctr_cutoff_pct_march_only": float(ctr_cutoff),
    "ctr_fix_signal_verdict": ctr_verdict,
    "queue_rows": int(len(ranked_queue)),
    "precision_at_k": {
        str(int(row.k)): float(row.precision_at_k)
        for row in precision_table.itertuples()
    },
}

metrics_path = OUTPUT_DIR / "baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Metrics receipt written to: {metrics_path}")


Overall future-decline base rate: 47.82%


,k,n_in_top_k,precision_at_k,precision_pct
0,10,10,0.500,50.0
1,50,50,0.480,48.0
2,100,100,0.450,45.0
3,500,500,0.522,52.2


Metrics receipt written to: work/outputs/baseline_metrics.json


## 3. Top-10 review

The assignment asks for the **top 10**. Each row gets:

- the action,
- why the rule placed it there,
- a confidence note,
- and what could make the recommendation wrong.

This review is written from the **March decision-time information**. April is not used to justify the action.


In [8]:
# Generate the required top-10 skeptical review.

top10 = ranked_queue.head(10).copy()

def confidence_for_reason(reason):
    if reason == "high_volume_ctr_fix":
        return "Higher: both rule conditions fire."
    if reason == "high_volume_only":
        return "Medium: volume is the only supporting signal."
    return "Low: baseline gives no action priority."

def wrong_if_for_reason(row):
    if row["reason_code"] == "high_volume_ctr_fix":
        return (
            "CTR may be low for a legitimate reason at this position, "
            "or the page may already match search intent well."
        )
    if row["reason_code"] == "high_volume_only":
        return (
            "High impressions may reflect stable demand; volume alone "
            "does not prove a content problem."
        )
    return (
        "The baseline is intentionally conservative and may miss a real "
        "opportunity that is not visible in these two signals."
    )

top10_review = top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "march_impressions",
        "march_ctr_pct",
        "march_avg_position",
    ]
].copy()

top10_review["why_its_here"] = top10_review["reason_code"].map(
    {
        "high_volume_ctr_fix": (
            "Both high-volume and CTR-fix conditions fired."
        ),
        "high_volume_only": (
            "March impressions reached the 500+ visibility threshold."
        ),
        "monitor": (
            "The row received no action-priority signal."
        ),
    }
)

top10_review["confidence_note"] = top10_review["reason_code"].map(
    confidence_for_reason
)

top10_review["what_would_make_it_wrong"] = top10_review.apply(
    wrong_if_for_reason,
    axis=1,
)

display(
    top10_review[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "why_its_here",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)
assert len(top10_review) == min(10, len(ranked_queue))


,rank,client_hash_id,content_hash_id,score,reason_code,action,why_its_here,confidence_note,what_would_make_it_wrong
0,1,client_62f4a7e64f5e0096,content_b99ea6861864dea5,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
1,2,client_73cda7b4e4f265ea,content_f43118e089ecc69a,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
2,3,client_62f4a7e64f5e0096,content_7c6373141eae744a,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
3,4,client_73cda7b4e4f265ea,content_95ff62babbfac9c7,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
4,5,client_23a62021009f63c4,content_5e1c049f62e33b11,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
5,6,client_73cda7b4e4f265ea,content_e578ac84778da489,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
6,7,client_62f4a7e64f5e0096,content_f6116743b00afc2d,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
7,8,client_e547b89c05043229,content_21309e9a83c83653,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
8,9,client_73cda7b4e4f265ea,content_cf651123f1085418,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...
9,10,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,5,high_volume_ctr_fix,CTR-fix review,Both high-volume and CTR-fix conditions fired.,Higher: both rule conditions fire.,CTR may be low for a legitimate reason at this...


## 4. Weak picks + leakage check

A strong baseline should survive skeptical review. A **weak pick** does not mean the score is mathematically wrong; it means the human rationale is easy to challenge.

The first weak-pick candidate is chosen from a `high_volume_only` row when available because volume alone is weaker evidence than the combined rule. If there is no such row, the notebook chooses the most borderline CTR-fix candidate.

After that, the notebook separately reports post-hoc false positives using April. Those outcomes are diagnostics only and never enter the score.


In [9]:
# Identify at least one rationale-weak candidate, then report post-hoc false positives.

weak_candidates = ranked_queue[
    ranked_queue["reason_code"].eq("high_volume_only")
].copy()

if len(weak_candidates) > 0:
    weak_pick = weak_candidates.head(1).copy()
    weak_pick["weakness"] = (
        "High volume alone is a weak rationale: the page has visibility, "
        "but the CTR-vs-position condition did not fire."
    )
else:
    ctr_candidates = ranked_queue[
        ranked_queue["reason_code"].eq("high_volume_ctr_fix")
    ].copy()

    weak_pick = ctr_candidates.sort_values(
        "march_ctr_pct",
        ascending=False,
        kind="mergesort",
    ).head(1).copy()

    weak_pick["weakness"] = (
        "This is a borderline CTR-fix pick because its CTR is closest to "
        "the March-only cutoff among the combined-signal items."
    )

print("Skeptical weak-pick review:")
display(
    weak_pick[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "march_impressions",
            "march_ctr_pct",
            "march_avg_position",
            "weakness",
        ]
    ]
)

# Post-hoc diagnostic only — never used in score construction.
ranked_with_outcome = ranked_queue.merge(
    evaluation_lookup,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one",
)

top10_posthoc = ranked_with_outcome.head(10)
false_positives = top10_posthoc[
    top10_posthoc["future_decline_label"].eq(0)
].copy()

print(
    f"Post-hoc false positives in top 10: "
    f"{len(false_positives)} / {len(top10_posthoc)}"
)

# Leakage guard: the committed queue must contain only decision-time fields.
forbidden_queue_columns = {
    "april_impressions",
    "future_decline_label",
    "trend_pct",
    "trend_direction",
    "is_declining",
    "leaked_decline_score",
}

assert forbidden_queue_columns.isdisjoint(set(ranked_queue.columns))
assert "client_name" not in ranked_queue.columns
assert "url" not in ranked_queue.columns
assert "query" not in ranked_queue.columns

print("PASS: no future-window or label-derived fields are present in the scored queue.")


Skeptical weak-pick review:


,rank,client_hash_id,content_hash_id,score,reason_code,action,march_impressions,march_ctr_pct,march_avg_position,weakness
18120,18121,client_e547b89c05043229,content_eadb33b5df496f4a,3,high_volume_only,Quick-win review,617124.0,0.918454,2.33147,High volume alone is a weak rationale: the pag...


Post-hoc false positives in top 10: 5 / 10
PASS: no future-window or label-derived fields are present in the scored queue.


## Self-check

- [x] Two signal checks have visible bucket tables with `n`.
- [x] At least one tested signal is linked to real FlyRank flag logic (CTR-vs-position / CTR-fix).
- [x] The rule is plain-English and transparent.
- [x] Score, reason code, and action label are generated from March-only information.
- [x] Ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Precision@K and the base rate are reported.
- [x] Top 10 rows are reviewed skeptically.
- [x] At least one weak rationale pick is surfaced.
- [x] April/future label is used only for post-hoc evaluation.
- [x] The scored CSV contains no future-window or label-derived inputs.
- [x] `work/outputs/baseline_metrics.json` is generated as a run receipt.
- [x] **Final user step:** Run **Runtime → Run all** in Colab with the real `HF_TOKEN`, confirm there are no errors, then commit this notebook under `work/notebooks/w04_baseline_score.ipynb`.

### Submission note

The CSV is intentionally a generated artifact and should stay out of Git. The notebook and the metrics JSON receipt are the useful committed evidence.
